## Models
| Model | Library | Notes |
|-------|---------|-------|
| xRFM | `xrfm` (official) | GPU-accelerated via CUDA 12 |
| XGBoost | `xgboost` | GPU-enabled (`tree_method='hist'`, `device='cuda'`) |
| Random Forest | `sklearn` | CPU-only baseline |

Hyperparameters are tuned on validation set (via xRFM's built-in tuning for xRFM; small grid search for XGBoost; defaults + sensible configs for RF). Final metrics are reported on the held-out test set only

In [28]:

import numpy as np
import pandas as pd
import os
import pickle
import time
import warnings
warnings.filterwarnings('ignore')


import torch
from xrfm import xRFM
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


from sklearn.metrics import (
    mean_squared_error, accuracy_score, roc_auc_score,
    root_mean_squared_error
)


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


DATASET_FILES = [
    'concrete',
    'energy',
    'bike_sharing',
    'online_shoppers',
    'shuttle',
]

datasets = {}
for name in DATASET_FILES:
    with open(f'processed/{name}.pkl', 'rb') as f:
        datasets[name] = pickle.load(f)
    d = datasets[name]
    print(f"  loaded {name}: n_train={d['n_train']}, d={d['n_features']}, task={d['task_type']}")

print(f"\n{len(datasets)} datasets loaded.")

Device: cuda
GPU: NVIDIA GeForce RTX 2060
  loaded concrete: n_train=659, d=8, task=regression
  loaded energy: n_train=491, d=8, task=regression
  loaded bike_sharing: n_train=11122, d=59, task=regression
  loaded online_shoppers: n_train=7891, d=75, task=classification
  loaded shuttle: n_train=37120, d=7, task=classification

5 datasets loaded.


In [29]:
def evaluate_regression(y_true, y_pred):
    """Compute RMSE for regression."""
    rmse = root_mean_squared_error(y_true, y_pred)
    return {'RMSE': rmse}


def evaluate_classification(y_true, y_pred_labels, y_pred_proba, n_classes):
    """Compute Accuracy and AUC-ROC for classification.
    Robust to cases where predicted proba matrix doesn't cover all classes."""
    acc = accuracy_score(y_true, y_pred_labels)
    labels = np.arange(n_classes)

    if n_classes == 2:
        if y_pred_proba.ndim == 2:
            proba_pos = y_pred_proba[:, 1]
        else:
            proba_pos = y_pred_proba
        auc = roc_auc_score(y_true, proba_pos)
    else:
        if y_pred_proba.shape[1] != n_classes:
            fixed = np.zeros((len(y_pred_proba), n_classes))
            fixed[:, :y_pred_proba.shape[1]] = y_pred_proba
            y_pred_proba = fixed / (fixed.sum(axis=1, keepdims=True) + 1e-12)
        auc = roc_auc_score(y_true, y_pred_proba, multi_class='ovr',
                            average='macro', labels=labels)
    return {'Accuracy': acc, 'AUC-ROC': auc}


def time_inference_per_sample(predict_fn, X_test, n_reps=3):
    """Measure average inference time per sample over n_reps runs."""
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        _ = predict_fn(X_test)
        t1 = time.perf_counter()
        times.append((t1 - t0) / len(X_test))
    return float(np.mean(times))


In [ ]:
import itertools

def train_xrfm(data, dataset_name):
    print(f"\n{'='*60}")
    print(f"Training xRFM on {dataset_name}")
    print(f"{'='*60}")

    task = data['task_type']
    n_train = data['n_train']

    X_train = torch.tensor(data['X_train'], dtype=torch.float32, device=DEVICE)
    X_val   = torch.tensor(data['X_val'],   dtype=torch.float32, device=DEVICE)
    X_test  = torch.tensor(data['X_test'],  dtype=torch.float32, device=DEVICE)

    if task == 'classification':
        y_train = torch.tensor(data['y_train'], dtype=torch.long, device=DEVICE)
        y_val   = torch.tensor(data['y_val'],   dtype=torch.long, device=DEVICE)
        tuning_metric = 'accuracy'
    else:
        y_train = torch.tensor(data['y_train'].reshape(-1, 1), dtype=torch.float32, device=DEVICE)
        y_val   = torch.tensor(data['y_val'].reshape(-1, 1),   dtype=torch.float32, device=DEVICE)
        tuning_metric = 'mse'

    # Grid ranges from the xRFM paper benchmarks: (CAN USE SMALLER RANGE FOR FASTER TRAINING)
    #   bandwidth: log-uniform [0.5, 200]
    #   exponent : uniform     [0.7, 1.4]
    #   reg      : log-uniform [1e-6, 1]
    #   iters    : fixed at 10
    _bandwidths = [0.5, 2.0, 10.0, 50.0, 200.0]
    _exponents  = [0.7, 1.0, 1.4]
    _regs       = [1e-6, 1e-4, 1e-2, 1.0]
    _iters      = [10]

    param_grid = [
        {'bandwidth': bw, 'exponent': exp, 'reg': reg, 'iters': itr}
        for bw, exp, reg, itr in itertools.product(_bandwidths, _exponents, _regs, _iters)
    ]
    print(f"  Grid size: {len(param_grid)} combos")

    from xrfm.rfm_src.gpu_utils import memory_scaling_factor as _gpu_scale
    import math
    _scale = _gpu_scale(DEVICE, quadratic=True)
    desired_leaf = max(4000, n_train // 4)
    max_leaf = math.ceil(desired_leaf / _scale) if _scale > 0 else desired_leaf

    best_score = None
    best_params = None
    best_model = None

    t0 = time.perf_counter()

    for params in param_grid:
        rfm_params = {
            'model': {
                'kernel': 'l2',
                'bandwidth': params['bandwidth'],
                'exponent': params['exponent'],
                'diag': False,
                'bandwidth_mode': 'adaptive',
            },
            'fit': {
                'reg': params['reg'],
                'iters': params['iters'],
                'early_stop_rfm': False,
            },
        }

        m = xRFM(
            rfm_params=rfm_params,
            max_leaf_size=max_leaf,
            device=DEVICE,
            tuning_metric=tuning_metric,
            verbose=False,
        )
        m.fit(X_train, y_train, X_val, y_val)

        if task == 'regression':
            val_pred = m.predict(X_val).ravel()
            score = root_mean_squared_error(data['y_val'], val_pred)
            is_better = (best_score is None) or (score < best_score)
        else:
            val_proba = m.predict_proba(X_val)
            if data['n_classes'] == 2:
                score = roc_auc_score(data['y_val'], val_proba[:, 1])
            else:
                score = accuracy_score(data['y_val'], val_proba.argmax(axis=1))
            is_better = (best_score is None) or (score > best_score)

        print(f"  bw={params['bandwidth']:5.1f}, exp={params['exponent']:.1f}, reg={params['reg']:.0e}, iters={params['iters']:2d} → val {tuning_metric}: {score:.4f}")

        if is_better:
            best_score = score
            best_params = params
            best_model = m

    train_time = time.perf_counter() - t0
    print(f"  Best: {best_params} | val score: {best_score:.4f}")
    print(f"  Training time (incl. tuning): {train_time:.2f}s")

    if task == 'regression':
        y_pred = best_model.predict(X_test).ravel()
        metrics = evaluate_regression(data['y_test'], y_pred)
    else:
        y_pred_labels = best_model.predict(X_test)
        y_pred_proba  = best_model.predict_proba(X_test)
        metrics = evaluate_classification(data['y_test'], y_pred_labels, y_pred_proba, data['n_classes'])

    inf_time = time_inference_per_sample(lambda X: best_model.predict(X), X_test, n_reps=3)

    result = {
        'dataset': dataset_name,
        'model': 'xRFM',
        'task': task,
        **metrics,
        'train_time_s': train_time,
        'inference_time_per_sample_s': inf_time,
        'n_train': data['n_train'],
        'n_features': data['n_features'],
    }

    metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  Test metrics: {metric_str}")
    print(f"  Inference time/sample: {inf_time*1000:.4f} ms")
    return result, best_model


In [31]:

all_results = []
xrfm_models = {}

for name in DATASET_FILES:
    try:
        result, model = train_xrfm(datasets[name], name)
        all_results.append(result)
        xrfm_models[name] = model
    except Exception as e:
        print(f"\n⚠️  xRFM FAILED on {name}: {type(e).__name__}: {e}")
        print(f"Skipping this dataset for xRFM; continuing with others.\n")


os.makedirs('models', exist_ok=True)
for name, model in xrfm_models.items():
    with open(f'models/xrfm_{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"saved xrfm_{name}.pkl")

print(f"\nxRFM trained on {len(xrfm_models)}/{len(DATASET_FILES)} datasets.")
print(f"\tResults collected so far: {len(all_results)}")


Training xRFM on concrete
  Grid size: 60 combos
  bw=  0.5, exp=0.7, reg=1e-06, iters=10 → val mse: 5.8405
  bw=  0.5, exp=0.7, reg=1e-04, iters=10 → val mse: 5.3578
  bw=  0.5, exp=0.7, reg=1e-02, iters=10 → val mse: 5.3508
  bw=  0.5, exp=0.7, reg=1e+00, iters=10 → val mse: 6.3202
  bw=  0.5, exp=1.0, reg=1e-06, iters=10 → val mse: 4.7604
  bw=  0.5, exp=1.0, reg=1e-04, iters=10 → val mse: 5.4308
  bw=  0.5, exp=1.0, reg=1e-02, iters=10 → val mse: 5.4349
  bw=  0.5, exp=1.0, reg=1e+00, iters=10 → val mse: 6.6080
  bw=  0.5, exp=1.4, reg=1e-06, iters=10 → val mse: 5.4878
  bw=  0.5, exp=1.4, reg=1e-04, iters=10 → val mse: 5.3688
  bw=  0.5, exp=1.4, reg=1e-02, iters=10 → val mse: 5.2632
  bw=  0.5, exp=1.4, reg=1e+00, iters=10 → val mse: 7.0586
  bw=  2.0, exp=0.7, reg=1e-06, iters=10 → val mse: 5.7256
  bw=  2.0, exp=0.7, reg=1e-04, iters=10 → val mse: 5.2361
  bw=  2.0, exp=0.7, reg=1e-02, iters=10 → val mse: 5.1918
  bw=  2.0, exp=0.7, reg=1e+00, iters=10 → val mse: 6.5871
  bw= 

In [32]:
print("xRFM models in cache:", list(xrfm_models.keys()))
print("\nxRFM results in all_results:")
for r in all_results:
    if r['model'] == 'xRFM':
        metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in r.items() if k in ['RMSE', 'Accuracy', 'AUC-ROC'])
        print(f"  {r['dataset']}: {metrics_str}")
        

xRFM models in cache: ['concrete', 'energy', 'bike_sharing', 'online_shoppers', 'shuttle']

xRFM results in all_results:
  concrete: RMSE=6.4349
  energy: RMSE=0.5075
  bike_sharing: RMSE=65.8898
  online_shoppers: Accuracy=0.8994, AUC-ROC=0.9175
  shuttle: Accuracy=0.9978, AUC-ROC=0.9805


In [ ]:
def train_xgboost(data, dataset_name):
    """
    Train XGBoost with light validation-based tuning.

    Uses GPU acceleration via `device='cuda'`, hist-based tree method.
    Small grid search over tree depth and learning rate using the validation set.
    """
    print(f"\n{'='*60}")
    print(f"Training XGBoost on {dataset_name}")
    print(f"{'='*60}")

    task = data['task_type']
    X_train = data['X_train']
    X_val   = data['X_val']
    X_test  = data['X_test']
    y_train = data['y_train']
    y_val   = data['y_val']
    y_test  = data['y_test']

    # Define a small hyperparameter grid
    depth_grid = [4, 6, 8]
    lr_grid = [0.05, 0.1]
    n_estimators_cap = 500

    # Common kwargs
    base_kwargs = dict(
        tree_method='hist',
        device='cuda' if torch.cuda.is_available() else 'cpu',
        n_estimators=n_estimators_cap,
        random_state=RANDOM_SEED,
        verbosity=0,
        early_stopping_rounds=20,
    )

    if task == 'regression':
        Model = xgb.XGBRegressor
        base_kwargs['objective'] = 'reg:squarederror'
    else:
        n_classes = data['n_classes']
        Model = xgb.XGBClassifier
        if n_classes == 2:
            base_kwargs['objective'] = 'binary:logistic'
            base_kwargs['eval_metric'] = 'auc'
        else:
            base_kwargs['objective'] = 'multi:softprob'
            base_kwargs['eval_metric'] = 'mlogloss'
            base_kwargs['num_class'] = n_classes 

    
    t0 = time.perf_counter()

    best_score = None
    best_params = None
    best_model = None

    for depth in depth_grid:
        for lr in lr_grid:
            kw = dict(base_kwargs, max_depth=depth, learning_rate=lr)
            m = Model(**kw)
            m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

           
            if task == 'regression':
                val_pred = m.predict(X_val)
                from sklearn.metrics import root_mean_squared_error
                score = root_mean_squared_error(y_val, val_pred)
                is_better = (best_score is None) or (score < best_score)
            else:
                if n_classes == 2:
                    val_proba = m.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, val_proba)
                else:
                    val_pred = m.predict(X_val)
                    score = accuracy_score(y_val, val_pred)
                is_better = (best_score is None) or (score > best_score)

            if is_better:
                best_score = score
                best_params = {'max_depth': depth, 'learning_rate': lr}
                best_model = m

    train_time = time.perf_counter() - t0
    print(f"  Training time (incl. tuning): {train_time:.2f}s")
    print(f"  Best params: {best_params} | val score: {best_score:.4f}")

   
    def predict_fn(X):
        return best_model.predict(X)

    if task == 'regression':
        y_pred = best_model.predict(X_test)
        metrics = evaluate_regression(y_test, y_pred)
    else:
        y_pred_labels = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)
        metrics = evaluate_classification(y_test, y_pred_labels, y_pred_proba, n_classes)

    inf_time_per_sample = time_inference_per_sample(predict_fn, X_test, n_reps=3)

    result = {
        'dataset': dataset_name,
        'model': 'XGBoost',
        'task': task,
        **metrics,
        'train_time_s': train_time,
        'inference_time_per_sample_s': inf_time_per_sample,
        'n_train': data['n_train'],
        'n_features': data['n_features'],
    }

    metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  Test metrics: {metric_str}")
    print(f"  Inference time/sample: {inf_time_per_sample*1000:.4f} ms")
    return result, best_model


In [34]:

xgb_models = {}

for name in DATASET_FILES:
    try:
        result, model = train_xgboost(datasets[name], name)
       
        all_results = [r for r in all_results if not (r['dataset']==name and r['model']=='XGBoost')]
        all_results.append(result)
        xgb_models[name] = model
    except Exception as e:
        print(f"\n⚠️  XGBoost FAILED on {name}: {type(e).__name__}: {e}")


for name, model in xgb_models.items():
    with open(f'models/xgb_{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"  saved xgb_{name}.pkl")

print(f"\nXGBoost coverage: {len(xgb_models)}/{len(DATASET_FILES)}")
print(f"Total results in registry: {len(all_results)}")


Training XGBoost on concrete
  Training time (incl. tuning): 2.52s
  Best params: {'max_depth': 6, 'learning_rate': 0.1} | val score: 4.8154
  Test metrics: RMSE=5.1542
  Inference time/sample: 0.0067 ms

Training XGBoost on energy
  Training time (incl. tuning): 4.05s
  Best params: {'max_depth': 4, 'learning_rate': 0.1} | val score: 0.4065
  Test metrics: RMSE=0.3542
  Inference time/sample: 0.0155 ms

Training XGBoost on bike_sharing
  Training time (incl. tuning): 7.03s
  Best params: {'max_depth': 4, 'learning_rate': 0.1} | val score: 68.8022
  Test metrics: RMSE=68.4992
  Inference time/sample: 0.0016 ms

Training XGBoost on online_shoppers
  Training time (incl. tuning): 1.51s
  Best params: {'max_depth': 6, 'learning_rate': 0.05} | val score: 0.9364
  Test metrics: Accuracy=0.9027, AUC-ROC=0.9299
  Inference time/sample: 0.0018 ms

Training XGBoost on shuttle
  Training time (incl. tuning): 18.93s
  Best params: {'max_depth': 4, 'learning_rate': 0.05} | val score: 0.9988
  Tes

In [35]:
def train_random_forest(data, dataset_name):
    """
    Train Random Forest with light validation-based tuning over n_estimators and max_depth.
    Uses all CPU cores via n_jobs=-1.
    """
    print(f"\n{'='*60}")
    print(f"Training Random Forest on {dataset_name}")
    print(f"{'='*60}")

    task = data['task_type']
    X_train = data['X_train']
    X_val   = data['X_val']
    X_test  = data['X_test']
    y_train = data['y_train']
    y_val   = data['y_val']
    y_test  = data['y_test']

    n_est_grid = [100, 300]
    max_depth_grid = [None, 20]

    Model = RandomForestRegressor if task == 'regression' else RandomForestClassifier

    t0 = time.perf_counter()

    best_score = None
    best_params = None
    best_model = None

    for n_est in n_est_grid:
        for md in max_depth_grid:
            m = Model(
                n_estimators=n_est,
                max_depth=md,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )
            m.fit(X_train, y_train)

            if task == 'regression':
                val_pred = m.predict(X_val)
                score = root_mean_squared_error(y_val, val_pred)
                is_better = (best_score is None) or (score < best_score)
            else:
                n_classes = data['n_classes']
                if n_classes == 2:
                    val_proba = m.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, val_proba)
                else:
                    val_pred = m.predict(X_val)
                    score = accuracy_score(y_val, val_pred)
                is_better = (best_score is None) or (score > best_score)

            if is_better:
                best_score = score
                best_params = {'n_estimators': n_est, 'max_depth': md}
                best_model = m

    train_time = time.perf_counter() - t0
    print(f"  Training time (incl. tuning): {train_time:.2f}s")
    print(f"  Best params: {best_params} | val score: {best_score:.4f}")

    def predict_fn(X):
        return best_model.predict(X)

    if task == 'regression':
        y_pred = best_model.predict(X_test)
        metrics = evaluate_regression(y_test, y_pred)
    else:
        n_classes = data['n_classes']
        y_pred_labels = best_model.predict(X_test)
        y_pred_proba = best_model.predict_proba(X_test)
        metrics = evaluate_classification(y_test, y_pred_labels, y_pred_proba, n_classes)

    inf_time_per_sample = time_inference_per_sample(predict_fn, X_test, n_reps=3)

    result = {
        'dataset': dataset_name,
        'model': 'RandomForest',
        'task': task,
        **metrics,
        'train_time_s': train_time,
        'inference_time_per_sample_s': inf_time_per_sample,
        'n_train': data['n_train'],
        'n_features': data['n_features'],
    }

    metric_str = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"  Test metrics: {metric_str}")
    print(f"  Inference time/sample: {inf_time_per_sample*1000:.4f} ms")
    return result, best_model



In [36]:
rf_models = {}

for name in DATASET_FILES:
    try:
        result, model = train_random_forest(datasets[name], name)
        all_results = [r for r in all_results if not (r['dataset']==name and r['model']=='RandomForest')]
        all_results.append(result)
        rf_models[name] = model
    except Exception as e:
        print(f"\n⚠️  Random Forest FAILED on {name}: {type(e).__name__}: {e}")

for name, model in rf_models.items():
    with open(f'models/rf_{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f"  saved rf_{name}.pkl")

print(f"\n✓ Random Forest coverage: {len(rf_models)}/{len(DATASET_FILES)}")
print(f"  Total results in registry: {len(all_results)}")


Training Random Forest on concrete
  Training time (incl. tuning): 1.14s
  Best params: {'n_estimators': 300, 'max_depth': 20} | val score: 4.9152
  Test metrics: RMSE=6.0390
  Inference time/sample: 0.2565 ms

Training Random Forest on energy
  Training time (incl. tuning): 1.03s
  Best params: {'n_estimators': 100, 'max_depth': None} | val score: 0.5463
  Test metrics: RMSE=0.5352
  Inference time/sample: 0.2087 ms

Training Random Forest on bike_sharing
  Training time (incl. tuning): 7.20s
  Best params: {'n_estimators': 300, 'max_depth': None} | val score: 73.5375
  Test metrics: RMSE=71.7062
  Inference time/sample: 0.0246 ms

Training Random Forest on online_shoppers
  Training time (incl. tuning): 1.99s
  Best params: {'n_estimators': 300, 'max_depth': None} | val score: 0.9263
  Test metrics: Accuracy=0.8978, AUC-ROC=0.9148
  Inference time/sample: 0.0349 ms

Training Random Forest on shuttle
  Training time (incl. tuning): 2.93s
  Best params: {'n_estimators': 100, 'max_dept

In [37]:
results_df = pd.DataFrame(all_results)

results_df = results_df[[
    'dataset', 'model', 'task',
    'RMSE', 'Accuracy', 'AUC-ROC',
    'train_time_s', 'inference_time_per_sample_s',
    'n_train', 'n_features'
]]

dataset_order = DATASET_FILES
model_order = ['xRFM', 'XGBoost', 'RandomForest']
results_df['dataset'] = pd.Categorical(results_df['dataset'], categories=dataset_order, ordered=True)
results_df['model'] = pd.Categorical(results_df['model'], categories=model_order, ordered=True)
results_df = results_df.sort_values(['dataset', 'model']).reset_index(drop=True)

results_df.to_csv('results/tables/model_results.csv', index=False)
print(f"Saved: results/tables/model_results.csv ({len(results_df)} rows)")
print()
print(results_df.to_string(index=False))


Saved: results/tables/model_results.csv (15 rows)

        dataset        model           task      RMSE  Accuracy  AUC-ROC  train_time_s  inference_time_per_sample_s  n_train  n_features
       concrete         xRFM     regression  6.434946       NaN      NaN      4.982895                     0.000004      659           8
       concrete      XGBoost     regression  5.154179       NaN      NaN      2.515969                     0.000007      659           8
       concrete RandomForest     regression  6.039030       NaN      NaN      1.141546                     0.000256      659           8
         energy         xRFM     regression  0.507513       NaN      NaN      4.359190                     0.000006      491           8
         energy      XGBoost     regression  0.354208       NaN      NaN      4.049815                     0.000015      491           8
         energy RandomForest     regression  0.535221       NaN      NaN      1.031977                     0.000209      491   

In [38]:
def build_pivot(results_df, metric):
    sub = results_df[['dataset', 'model', metric]].copy()
    pivot = sub.pivot(index='dataset', columns='model', values=metric)
    pivot = pivot.reindex(index=dataset_order, columns=model_order)
    return pivot

print("=" * 70)
print("RMSE (regression, lower is better)")
print("=" * 70)
rmse_pivot = build_pivot(results_df[results_df['task']=='regression'], 'RMSE')
print(rmse_pivot.round(4).to_string())

print("\n" + "=" * 70)
print("Accuracy (classification, higher is better)")
print("=" * 70)
acc_pivot = build_pivot(results_df[results_df['task']=='classification'], 'Accuracy')
print(acc_pivot.round(4).to_string())

print("\n" + "=" * 70)
print("AUC-ROC (classification, higher is better)")
print("=" * 70)
auc_pivot = build_pivot(results_df[results_df['task']=='classification'], 'AUC-ROC')
print(auc_pivot.round(4).to_string())

print("\n" + "=" * 70)
print("Training time (seconds)")
print("=" * 70)
train_pivot = build_pivot(results_df, 'train_time_s')
print(train_pivot.round(2).to_string())

print("\n" + "=" * 70)
print("Inference time per sample (milliseconds)")
print("=" * 70)
inf_pivot = build_pivot(results_df, 'inference_time_per_sample_s') * 1000
print(inf_pivot.round(4).to_string())

rmse_pivot.to_csv('results/tables/pivot_rmse.csv')
acc_pivot.to_csv('results/tables/pivot_accuracy.csv')
auc_pivot.to_csv('results/tables/pivot_auc.csv')
train_pivot.to_csv('results/tables/pivot_train_time.csv')
(inf_pivot).to_csv('results/tables/pivot_inference_time_ms.csv')

print("\nAll pivot tables saved to results/tables/")

RMSE (regression, lower is better)
model               xRFM  XGBoost  RandomForest
dataset                                        
concrete          6.4349   5.1542        6.0390
energy            0.5075   0.3542        0.5352
bike_sharing     65.8898  68.4992       71.7062
online_shoppers      NaN      NaN           NaN
shuttle              NaN      NaN           NaN

Accuracy (classification, higher is better)
model              xRFM  XGBoost  RandomForest
dataset                                       
concrete            NaN      NaN           NaN
energy              NaN      NaN           NaN
bike_sharing        NaN      NaN           NaN
online_shoppers  0.8994   0.9027        0.8978
shuttle          0.9978   0.9983        0.9987

AUC-ROC (classification, higher is better)
model              xRFM  XGBoost  RandomForest
dataset                                       
concrete            NaN      NaN           NaN
energy              NaN      NaN           NaN
bike_sharing        NaN